# BLIP Image Captioning Example

## BLIP — Image Captioning

This notebook is an example of image captioning with **BLIP (Bootstrapping Language-Image Pre-training)**.

```
Image (URL or file) → BLIP model → Natural language caption
```

BLIP generates a natural language description of the entire image — what objects are present, what actions are happening, and the overall scene context.

### Available models

| Model | Description | Speed |
|-------|-------------|-------|
| `Salesforce/blip-image-captioning-large` | Larger, more descriptive captions | slower |
| `Salesforce/blip-image-captioning-base`  | Faster, good quality captions | faster |

Change `MODEL_NAME` in the writefile cell to switch between them.

📄 [BLIP paper](https://arxiv.org/abs/2201.12086)  
🤗 [Hugging Face model hub](https://huggingface.co/Salesforce/blip-image-captioning-large)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/CV-Samples/blip"
!mkdir -p "{PROJECT_PATH}"
%cd "{PROJECT_PATH}"
!ls


In [ ]:
# Download sample images from GitHub
import os

BASE_URL = "https://raw.githubusercontent.com/mastnk/cv-samples/main/blip"
IMAGE_FILES = [
    "furgao52-monkey-9040130_640.jpg",
]
for fname in IMAGE_FILES:
    url  = f'{BASE_URL}/{fname}'
    dest = f'{PROJECT_PATH}/{fname}'
    if not os.path.exists(dest):
        os.system(f'wget -q "{url}" -O "{dest}"')
        print(f'Downloaded: {fname}')
    else:
        print(f'Already exists: {fname}')

%cd "{PROJECT_PATH}"
!ls


In [ ]:
# Install dependencies
!pip install transformers accelerate -q

import transformers
print(transformers.__version__)  # check version


## Using Your Own Images

There are two ways to provide images for captioning:

**① Specify a URL**  
Pass a direct image URL to the `--url` flag when running the script:
```
%run blip.py --url https://cdn.pixabay.com/photo/2022/09/09/03/42/thailand-7442285_640.jpg
```

**② Upload images to the `PROJECT_PATH/` folder**  
Place your image files in `PROJECT_PATH/`, then run the script with `--file` or `--dir`.

The easiest way to upload is through **Google Drive**:  
Open [drive.google.com](https://drive.google.com), navigate to `My Drive / CV-Samples / blip/`, and drag & drop your files there.  
They will be immediately available in Colab via the mounted drive — no extra upload step needed.

```
%run blip.py --file your_image.jpg   # single file
%run blip.py --dir  .                # all images in folder
```

## Selecting a Model

In the writefile cell below, change `MODEL_NAME` to switch models.  
When multiple `MODEL_NAME = ...` lines are listed, **only the last one takes effect**.

```python
# MODEL_NAME = 'Salesforce/blip-image-captioning-large'  # more descriptive, slower
MODEL_NAME   = 'Salesforce/blip-image-captioning-base'   # faster, good quality  ← active
```

The `large` model generates richer captions but takes longer to load and run.

In [ ]:
# Save this cell as a Python file (Execute after editing)
%%writefile blip.py
"""BLIP Image Captioning — command-line interface.

Usage:
  %run blip.py --url  <image_url>  [--disp]
  %run blip.py --file <image_path> [--disp]
  %run blip.py --dir  <image_dir>  [--disp]
"""

import argparse
import glob
import os

from PIL import Image
import requests
from io import BytesIO
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration

# ---- IPython display (works when executed via %run in Colab) -----
try:
    from IPython.display import display as ipy_display
    _has_ipy = True
except ImportError:
    _has_ipy = False

# ---- Configuration -----------------------------------------------
MODEL_NAME = 'Salesforce/blip-image-captioning-base'   # faster, good quality
# MODEL_NAME = 'Salesforce/blip-image-captioning-large'  # more descriptive, slower

PROJECT_PATH = '/content/drive/MyDrive/CV-Samples/blip'

# ---- Model loading -----------------------------------------------
device    = 'cuda' if torch.cuda.is_available() else 'cpu'
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model     = BlipForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    tie_word_embeddings=False,
).to(device)

# ---- Caption helper ----------------------------------------------
def generate_caption(image: Image.Image) -> str:
    """Generate a caption for a PIL image."""
    inputs = processor(image, return_tensors='pt').to(device)
    out    = model.generate(**inputs)
    return processor.decode(out[0], skip_special_tokens=True)

# ---- Display helper ----------------------------------------------
def show(image: Image.Image, label: str, disp: bool) -> None:
    """When --disp is active, display the image then print the filename/URL."""
    if disp:
        if _has_ipy:
            ipy_display(image)
        print(label)

# ---- Functions ---------------------------------------------------
def caption_url(url: str, disp: bool = False):
    """Download an image from a URL and generate a caption."""
    image   = Image.open(BytesIO(requests.get(url).content)).convert('RGB')
    show(image, url, disp)
    caption = generate_caption(image)
    print(f'  Caption: {caption}')
    return caption


def caption_file(path: str, disp: bool = False):
    """Generate a caption for a single local image file."""
    image   = Image.open(path).convert('RGB')
    show(image, path, disp)
    caption = generate_caption(image)
    print(f'  Caption: {caption}')
    return caption


def caption_dir(directory: str, disp: bool = False):
    """Generate captions for all images in a directory."""
    exts = ['jpg', 'jpeg', 'png', 'bmp', 'JPG', 'JPEG', 'PNG', 'BMP']
    filepaths = []
    for ext in exts:
        filepaths += sorted(glob.glob(os.path.join(directory, f'*.{ext}')))

    if not filepaths:
        print(f'No images found in: {directory}')
        return

    for path in filepaths:
        caption_file(path, disp)
        print()


# ---- Argument parsing --------------------------------------------
parser = argparse.ArgumentParser(description='BLIP Image Captioning')
group  = parser.add_mutually_exclusive_group(required=True)
group.add_argument('--url',  type=str, help='Image URL to caption')
group.add_argument('--file', type=str, help='Path to a single image file')
group.add_argument('--dir',  type=str, help='Directory of images to caption')
parser.add_argument('--disp', action='store_true',
                    help='Display image and filename/URL before caption')
args = parser.parse_args()

# ---- Run ---------------------------------------------------------
if args.url:
    caption_url(args.url,   disp=args.disp)
elif args.file:
    caption_file(args.file, disp=args.disp)
elif args.dir:
    caption_dir(args.dir,   disp=args.disp)


## `blip.py` Usage

After running the `%%writefile` cell above, `blip.py` is saved in `PROJECT_PATH`.
Run it with **`%run`** (not `!python`) so that inline image display works in Colab.

```
%run blip.py --url  <image_url>        # generate caption for a remote image
%run blip.py --file <image_path>       # generate caption for a single local file
%run blip.py --dir  <image_directory>  # generate captions for all images in a folder
```

**Optional arguments**

| Flag | Default | Description |
|------|---------|-------------|
| `--disp` | off | Display image and filename / URL before caption |

**Examples**

```bash
# Generate caption from URL and display image
%run blip.py --url https://cdn.pixabay.com/photo/2022/09/09/03/42/thailand-7442285_640.jpg --disp

# Generate caption from a local file
%run blip.py --file furgao52-monkey-9040130_640.jpg --disp

# Generate captions for every image in the folder
%run blip.py --dir . --disp
```

**Output format (with `--disp`)**

```
<image displayed inline>
furgao52-monkey-9040130_640.jpg
  Caption: a monkey sitting on a branch in a tree
```

## Execution Methods

Use **`%run`** to execute `blip.py` inside the Colab kernel — this enables inline image display with `--disp`.

| Mode | Flag | Description |
|------|------|-------------|
| From URL | `--url <url>` | Fetches and captions a remote image |
| Single file | `--file <path>` | Captions one local image |
| Directory | `--dir <path>` | Captions all images in a folder |

Add `--disp` to display each image and its filename / URL before the caption.

In [ ]:
# Execute: generate caption from URL (with display)
%cd "{PROJECT_PATH}"
%run blip.py --disp --url https://cdn.pixabay.com/photo/2022/09/09/03/42/thailand-7442285_640.jpg


In [ ]:
# Execute: generate caption from a single local image file (with display)
%cd "{PROJECT_PATH}"
%run blip.py --disp --file furgao52-monkey-9040130_640.jpg


In [ ]:
# Execute: generate captions for all images in a directory (with display)
%cd "{PROJECT_PATH}"
%run blip.py --disp --dir .


💾 **Don't forget to save this notebook!**

To keep your work, go to **File → Save a copy in Drive** before closing.